# Hungarian Algorithm in Multi-object Tracking
The hungarian algorithm is used to associate the predictions with the actual measurment and assgin bounding boxes accordingly. The general domain independend target is to search the minimal sum of all it values. As the following:

$$
\min \sum_{j=1}^n c_{{s_j}j}
$$

In [1]:
import numpy as np

## Build the Cost Matrix
To actually track, so to distinguish between various detections we need to assign a prediction coming from our kalman filter *where an object is?* and *what is measured?* Parts of *What is measured?* Already infuence our prediction using the Kalman-filter to detections coming from our detection system. Therefor we need to build a combination matrix where the $i$ entries correspond to our trackings and the $j$ entries correspond to our detections. We assign a cost to each of the various combination. Assigning costs mostly depends on the metrics we want to compute, common metrics are:

- Mahalanobis distance
- 1 − $IoU$
- Motion + appearance cost


## Row Reduction
Now the first step of the actual algorihm starts, which is indepened from the domain we apply the algorithm to. Inside the cost matrix we will select the smallest element and substract it from the given row. Considering $C$ as our cost matrix.

$$
\mathbf{\hat c_i} = c_i - \min c_i 
$$

In [2]:
def row_reduction(cost_mat):
    dim = cost_mat.shape[0]
    cur_mat = mat

    for row_num in range(dim): 
        cur_mat[row_num] = cur_mat[row_num] - np.min(cur_mat[row_num])
    return cur_mat

## Column Reduction
The next step is acutally the same again as for row-reduction but this time we reduce the column $j$ of the cost matrix $C$.
$$
\mathbf{\hat c_j} = c_j - \min c_j
$$

In [3]:
def col_reduction(cost_mat):
    dim = cost_mat.shape[0]
    cur_mat = mat
    
    for col_num in range(dim): 
        cur_mat[:,col_num] = cur_mat[:,col_num] - np.min(cur_mat[:,col_num])
    return cur_mat

## Cover All Zeros with a Minimum Number of Lines
Cover all zeros in the resulting matrix using the minimum number of horizontal and vertical lines. So we track all zeros resulting from our substraction. 

In [4]:
def min_zero_row(zero_mat, mark_zero):
    min_row = [99999, -1]

    for row_num in range(zero_mat.shape[0]): 
        if np.sum(zero_mat[row_num] == True) > 0 and min_row[0] > np.sum(zero_mat[row_num] == True):
            min_row = [np.sum(zero_mat[row_num] == True), row_num]
            
    zero_index = np.where(zero_mat[min_row[1]] == True)[0][0]
    mark_zero.append((min_row[1], zero_index))
    zero_mat[min_row[1], :] = False
    zero_mat[:, zero_index] = False
    

The function `mark_matrix` is mainly used to mark the columns and rows from the matrix.

In [5]:
def mark_matrix(mat):
    cur_mat = mat
    zero_bool_mat = (cur_mat == 0)
    zero_bool_mat_copy = zero_bool_mat.copy()

    marked_zero = []
    
    while (True in zero_bool_mat_copy):
        min_zero_row(zero_bool_mat_copy, marked_zero)

    marked_zero_row = []
    marked_zero_col = []
    
    for i in range(len(marked_zero)):
        marked_zero_row.append(marked_zero[i][0])
        marked_zero_col.append(marked_zero[i][1])
    non_marked_row = list(set(range(cur_mat.shape[0])) - set(marked_zero_row))
    
    marked_cols = []
    check_switch = True
    
    while check_switch:
        check_switch = False
        
        for i in range(len(non_marked_row)):
            row_array = zero_bool_mat[non_marked_row[i], :]
            
            for j in range(row_array.shape[0]):
                if row_array[j] == True and j not in marked_cols:
                    marked_cols.append(j)
                    check_switch = True

        for row_num, col_num in marked_zero:
            if row_num not in non_marked_row and col_num in marked_cols:
                non_marked_row.append(row_num)
                check_switch = True
                
    marked_rows = list(set(range(mat.shape[0])) - set(non_marked_row))
    return(marked_zero, marked_rows, marked_cols)

The `adjust_matrix` function is only executed whenever we don't mark equal number of "zero lines" as the matrix as number of rows and columns. In this case we subtract the smallest uncovered entry  from all uncovered rows. Then we add the smallest entry to all covered columns, we get. We repeat until we get the $n$ numbers of lines. Afterwards

In [6]:
def adjust_matrix(mat, cover_rows, cover_cols):
	cur_mat = mat
	non_zero_element = []

	for row in range(len(cur_mat)):
		if row not in cover_rows:
			for i in range(len(cur_mat[row])):
				if i not in cover_cols:
					non_zero_element.append(cur_mat[row][i])
	min_num = min(non_zero_element)

	for row in range(len(cur_mat)):
		if row not in cover_rows:
			for i in range(len(cur_mat[row])):
				if i not in cover_cols:
					cur_mat[row, i] = cur_mat[row, i] - min_num
                    
	for row in range(len(cover_rows)):  
		for col in range(len(cover_cols)):
			cur_mat[cover_rows[row], cover_cols[col]] = cur_mat[cover_rows[row], cover_cols[col]] + min_num
	return cur_mat

## Check Optimal
The optimal value is found by adding up the values of the index where found the zeros and assigned vertical and horizontal lines.

In [7]:
def ans_calculation(mat, pos):
	total = 0
	ans_mat = np.zeros((mat.shape[0], mat.shape[1]))
	for i in range(len(pos)):
		total += mat[pos[i][0], pos[i][1]]
		ans_mat[pos[i][0], pos[i][1]] = mat[pos[i][0], pos[i][1]]
	return total, ans_mat

In [8]:
def hungarian_algorithm(mat): 
	dim = mat.shape[0]
	cur_mat = mat

	#Step 1 - Every column and every row subtract its internal minimum
	for row_num in range(mat.shape[0]): 
		cur_mat[row_num] = cur_mat[row_num] - np.min(cur_mat[row_num])
	
	for col_num in range(mat.shape[1]): 
		cur_mat[:,col_num] = cur_mat[:,col_num] - np.min(cur_mat[:,col_num])
	zero_count = 0
	while zero_count < dim:
		#Step 2 & 3
		ans_pos, marked_rows, marked_cols = mark_matrix(cur_mat)
		zero_count = len(marked_rows) + len(marked_cols)

		if zero_count < dim:
			cur_mat = adjust_matrix(cur_mat, marked_rows, marked_cols)

	return ans_pos

In [9]:
def optimize():
	cost_matrix = np.array([[7, 6, 2, 9, 2],
				[6, 2, 1, 3, 9],
				[5, 6, 8, 9, 5],
				[6, 8, 5, 8, 6],
				[9, 5, 6, 4, 7]])
	ans_pos = hungarian_algorithm(cost_matrix.copy())#Get the element position.
	ans, ans_mat = ans_calculation(cost_matrix, ans_pos)#Get the minimum or maximum value and corresponding matrix.

	#Show the result
	print(f"Linear Assignment problem result: {ans:.0f}\n{ans_mat}")

	profit_matrix = np.array([[7, 6, 2, 9, 2],
				[6, 2, 1, 3, 9],
				[5, 6, 8, 9, 5],
				[6, 8, 5, 8, 6],
				[9, 5, 6, 4, 7]])
	max_value = np.max(profit_matrix)
	cost_matrix = max_value - profit_matrix
	ans_pos = hungarian_algorithm(cost_matrix.copy())#Get the element position.
	ans, ans_mat = ans_calculation(profit_matrix, ans_pos)#Get the minimum or maximum value and corresponding matrix.
	print(f"Linear Assignment problem result: {ans:.0f}\n{ans_mat}")

In [10]:
optimize()

Linear Assignment problem result: 18
[[0. 0. 0. 0. 2.]
 [0. 2. 0. 0. 0.]
 [5. 0. 0. 0. 0.]
 [0. 0. 5. 0. 0.]
 [0. 0. 0. 4. 0.]]
Linear Assignment problem result: 43
[[0. 0. 0. 9. 0.]
 [0. 0. 0. 0. 9.]
 [0. 0. 8. 0. 0.]
 [0. 8. 0. 0. 0.]
 [9. 0. 0. 0. 0.]]
